# Optimisation for Energy Planning: From Two Power Plants to a Capacity Expansion Model

**ENGG562/ENV462 · Climate Change Governance: Science, Data & Models · LUMS · Fall 2026**

**Module 2: Scenarios & Integrated Assessment** · Coding exercise 03 · First notebook of the *mini integrated assessment model* series · Requires coding exercise 01 or equivalent experience with Python

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LUMS-WIT/climate-futures/blob/main/content-src/coding/03-optimisation-for-energy-planning.ipynb)

Process-based integrated assessment models such as MESSAGEix, which was introduced in Lecture 9, are **optimisation models**. Given demands for energy services, a list of technologies with their costs and efficiencies, and a set of constraints, the model chooses the combination of investments and operating decisions that meets the demands at the lowest total cost. Every result that such a model reports, including the electricity mix, the carbon price, and the cost of a climate target, follows from that single calculation.

This notebook builds the idea from the ground up. It begins with a problem small enough to solve with pencil and paper, then solves the same problem with a computer, then extends it step by step until it has the structure of the MESSAGE energy model: several technologies, several periods, investment in capacity, technical lifetimes, discounting, and an emission constraint.

## The mini integrated assessment model series

| Notebook | Content |
|---|---|
| **03 (this notebook)** | Linear optimisation, shadow prices, and a small capacity expansion model |
| 04 | Westeros: the MESSAGE formulation on the teaching example of the MESSAGEix tutorials |
| 05 | World: the same formulation applied to a global energy system, with carbon budgets and warming |
| 06 | MESSAGE-MACRO: linking the energy model to a macro-economic growth model |
| 07 | From the mini model to the real `message_ix` package |

The concepts are introduced on the small example first and then reused, unchanged, on the larger one.

Three kinds of prompt appear throughout:

- **💬 Predict before running the next cell:** write a short prediction *before* running the cell that follows, then compare it with the result.
- **✏️ Exercise:** replace every `...` or `___` with code. Self-check cells print ✅ or ❌.
- **💬 Question:** a written answer of two to four sentences.

## Learning objectives

On completing this notebook, students should be able to:

1. Formulate a planning question as a **linear programme**: decision variables, objective function, and constraints.
2. Solve a linear programme graphically, with `scipy`, and with the algebraic modelling language **Pyomo**.
3. Interpret **shadow prices** (dual values) as the price of electricity and the price of carbon.
4. Explain why a capacity expansion model distinguishes **investment** in capacity from the **activity** of that capacity, and how lifetime and discounting enter the calculation.
5. Construct a **marginal abatement cost curve** from repeated model runs.

## Structure of the notebook

| Part | Content | Approximate time |
|---|---|---|
| 0 | Setup | 5 min |
| 1 | What is an optimisation model? | 10 min |
| 2 | A dispatch problem with two power plants | 20 min |
| 3 | Shadow prices: the price of electricity and the price of carbon | 20 min |
| 4 | Algebraic modelling with Pyomo | 20 min |
| 5 | A capacity expansion model over several decades | 30 min |
| 6 | A marginal abatement cost curve | 15 min |
| 7 | Summary: from this notebook to MESSAGE | 5 min |

---
# Part 0 · Setup

The cell below installs the modelling language Pyomo and the open-source solver HiGHS when the notebook runs in Google Colab. Installation takes less than one minute. On a personal computer, the same packages are installed once with `pip install pyomo highspy`.

In [ ]:
import sys

if "google.colab" in sys.modules:
    !pip install --quiet pyomo highspy

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linprog
import pyomo.environ as pyo

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
pd.set_option("display.precision", 2)

solver = pyo.SolverFactory("appsi_highs")
print("HiGHS solver available:", solver.available())

In [ ]:
def check(description, condition):
    """Self-check helper: prints a tick or a cross. It never stops the notebook."""
    try:
        passed = bool(condition())
    except Exception:
        passed = False
    print(("✅ " if passed else "❌ ") + description)

---
# Part 1 · What is an optimisation model?

Every optimisation model has three ingredients.

| Ingredient | Meaning | Example in an energy model |
|---|---|---|
| **Decision variables** | The quantities the model is free to choose | How much electricity each power plant generates; how much new capacity is built |
| **Objective function** | The quantity to be minimised or maximised | Total discounted cost of the energy system |
| **Constraints** | Conditions every solution must satisfy | Demand must be met; a plant cannot generate more than its capacity allows; emissions must stay below a limit |

When the objective and every constraint are **linear** in the decision variables (sums of variables multiplied by constants), the problem is a **linear programme (LP)**. Linear programmes with millions of variables can be solved reliably within minutes, which is why the energy part of MESSAGE, and of most process-based integrated assessment models, is formulated as an LP.

A least-cost model answers a normative question: *what would a perfectly informed central planner do?* It does not predict what markets, firms, or households will in fact do. This distinction matters whenever model results are used in policy debate, and it returns in the questions at the end of each part.

---
# Part 2 · A dispatch problem with two power plants

A small country (we will call it Westeros, after the teaching example of the MESSAGEix tutorials) needs **50 GWa** of electricity in one year. One GWa is the energy delivered by a 1 GW plant running for a whole year, which equals 8.76 TWh.

Two plants are available:

| Plant | Running cost (USD per kWa) | Maximum output (GWa) | CO₂ (t per kWa) |
|---|---|---|---|
| Coal | 60 | 45 | 8.0 |
| Wind | 20 | 30 | 0 |

The running cost of wind is low because wind needs no fuel; its output is limited by how much wind blows on the installed turbines.

Written as a linear programme, with $x_c$ and $x_w$ the output of coal and wind in GWa:

$$
\begin{aligned}
\min_{x_c, x_w} \quad & 60\,x_c + 20\,x_w && \text{(total cost, million USD)} \\
\text{subject to} \quad & x_c + x_w \geq 50 && \text{(demand)} \\
& x_c \leq 45, \quad x_w \leq 30 && \text{(availability)} \\
& x_c, x_w \geq 0
\end{aligned}
$$

(Units: GWa multiplied by USD/kWa gives million USD, because 1 GWa is one million kWa.)

**💬 Predict before running the next cell.** Which plant will the cheapest solution use first, and how much coal will be needed?

*Answer:* *(write the answer here)*

In [ ]:
# Draw the feasible region and lines of equal cost
xc = np.linspace(0, 60, 400)
fig, ax = plt.subplots()
ax.fill_between(xc, np.maximum(50 - xc, 0), 30, where=(xc <= 45) & (50 - xc <= 30),
                color="tab:green", alpha=0.2, label="feasible region")
ax.plot(xc, 50 - xc, color="black", label="demand: $x_c + x_w = 50$")
ax.axhline(30, color="tab:blue", ls="--", label="wind limit")
ax.axvline(45, color="tab:brown", ls="--", label="coal limit")
for cost in (1200, 1800, 2400):
    ax.plot(xc, (cost - 60 * xc) / 20, color="grey", lw=0.8, ls=":")
    ax.annotate(f"cost = {cost}", xy=(cost / 60 - 6, 6), color="grey", fontsize=8)
ax.plot(20, 30, "o", color="tab:red", ms=9, label="cheapest point")
ax.set(xlim=(0, 55), ylim=(0, 55), xlabel="coal output $x_c$ (GWa)", ylabel="wind output $x_w$ (GWa)")
ax.legend(loc="upper right", fontsize=8)
plt.show()

The dotted lines are combinations with equal total cost. Moving towards the origin lowers the cost, and the optimum is the last point of the feasible region that a line of equal cost touches. For a linear programme, the optimum always lies at a **corner** of the feasible region, which explains why linear models often produce "all or nothing" results (a point that returns in notebook 05).

The same problem is solved by computer with `scipy.optimize.linprog`, which expects constraints in the form $A_{ub}\,x \leq b_{ub}$. The demand constraint $x_c + x_w \geq 50$ is therefore written as $-x_c - x_w \leq -50$.

In [ ]:
cost = [60, 20]                     # objective coefficients: coal, wind
A_ub = [[-1, -1]]                   # -x_c - x_w <= -50  (demand)
b_ub = [-50]
bounds = [(0, 45), (0, 30)]         # availability limits

res = linprog(cost, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method="highs")
print(res.message)
print(f"coal = {res.x[0]:.1f} GWa, wind = {res.x[1]:.1f} GWa, total cost = {res.fun:,.0f} million USD")

### ✏️ Exercise 1: A calmer year

In a year with less wind, the wind farms can deliver only **18 GWa**. Solve the problem again and store the result in `res_calm`.

In [ ]:
bounds_calm = [(0, 45), (0, ___)]
res_calm = linprog(cost, A_ub=A_ub, b_ub=b_ub, bounds=bounds_calm, method="highs")
print(res_calm.x, res_calm.fun)

In [ ]:
check("coal output is 32 GWa", lambda: abs(res_calm.x[0] - 32) < 1e-6)
check("total cost is 2,280 million USD", lambda: abs(res_calm.fun - 2280) < 1e-6)

**💬 Question.** Reduce the wind limit further, to 4 GWa, and run the cell again. What does the solver report, and what does the result mean for a planner?

*Answer:* *(write the answer here)*

---
# Part 3 · Shadow prices: the price of electricity and the price of carbon

Every constraint of a linear programme has a **shadow price** (or **dual value**): the change in the optimal total cost if the right-hand side of that constraint were increased by one unit. Shadow prices are produced by the solver at no extra cost, and they are among the most useful outputs of an energy model.

- The shadow price of the **demand constraint** is the cost of supplying one more unit of electricity: the **marginal cost**, which in a competitive market would be the electricity price. MESSAGEix reports it as `PRICE_COMMODITY`.
- The shadow price of an **emission constraint** is the cost of reducing emissions by one more tonne: the **carbon price** implied by the constraint. MESSAGEix reports it as `PRICE_EMISSION`.

With the HiGHS method, `linprog` reports shadow prices in `res.ineqlin.marginals`. For a "less than or equal" constraint in a minimisation, the marginal is negative or zero, so the sign is reversed below.

In [ ]:
price = -res.ineqlin.marginals[0]
print(f"Shadow price of the demand constraint: {price:.0f} USD per kWa")

**💬 Predict before running the next cell.** Before checking the output: why is the electricity price equal to 60, the running cost of coal, and not the average cost of 36 (1,800 ÷ 50)?

*Answer:* *(write the answer here)*

### Adding a third plant and an emission limit

A gas plant is now added (running cost 80 USD/kWa, maximum 50 GWa, 3.4 t CO₂ per kWa), and the government sets an **emission limit of 100 Mt CO₂** for the year. One GWa multiplied by t/kWa gives Mt, so the emission constraint is $8.0\,x_c + 3.4\,x_g \leq 100$.

In [ ]:
#        coal  gas  wind
cost3 = [60,   80,  20]
A3 = [[-1,  -1,  -1],      # demand: -(x_c + x_g + x_w) <= -50
      [8.0, 3.4,  0]]      # emissions: 8.0 x_c + 3.4 x_g <= 100
b3 = [-50, 100]
bounds3 = [(0, 45), (0, 50), (0, 30)]

res3 = linprog(cost3, A_ub=A3, b_ub=b3, bounds=bounds3, method="highs")
coal, gas, wind = res3.x
elec_price, co2_price = -res3.ineqlin.marginals
print(f"coal {coal:.1f}, gas {gas:.1f}, wind {wind:.1f} GWa | cost {res3.fun:,.0f} million USD")
print(f"electricity price {elec_price:.1f} USD/kWa | carbon price {co2_price:.2f} USD/t CO2")

The carbon price can be checked by hand. To avoid one more tonne of CO₂, the planner replaces some coal with gas. Each kWa switched raises the cost by $80 - 60 = 20$ USD and lowers emissions by $8.0 - 3.4 = 4.6$ t. The cost per tonne avoided is therefore $20 / 4.6 \approx 4.35$ USD per tonne, which is the shadow price reported above.

### ✏️ Exercise 2: The carbon price by finite difference

Verify the meaning of the shadow price directly. Solve the problem again with an emission limit of **99** Mt instead of 100, store the result in `res99`, and compute `delta_cost`, the increase in total cost. Compare it with `co2_price`.

In [ ]:
b99 = [-50, ___]
res99 = linprog(cost3, A_ub=A3, b_ub=b99, bounds=bounds3, method="highs")
delta_cost = ___ - ___
print(f"cost increase for one tonne less: {delta_cost:.2f} | shadow price: {co2_price:.2f}")

In [ ]:
check("delta_cost equals the carbon price", lambda: abs(delta_cost - co2_price) < 1e-6)

### ✏️ Exercise 3: A carbon tax instead of a limit

Instead of an emission limit, the government levies a **carbon tax** equal to `co2_price` on every tonne emitted. Add the tax to the running cost of each plant (tax × emission factor), remove the emission constraint, and solve. Store the result in `res_tax`. Does the tax produce the same generation mix as the limit?

<details><summary>Hint</summary>

At a tax exactly equal to the shadow price, coal and gas have the same cost including the tax, so the solver is indifferent between them and may return either corner. A tax a fraction of a cent higher makes the switch unambiguous.

</details>

In [ ]:
ef = np.array([8.0, 3.4, 0.0])
cost_tax = np.array(cost3) + ___ * ef
res_tax = linprog(cost_tax, A_ub=[A3[0]], b_ub=[-50], bounds=bounds3, method="highs")
print(res_tax.x.round(2), "compared with", res3.x.round(2))

In [ ]:
check("the tax run meets demand", lambda: abs(res_tax.x.sum() - 50) < 1e-6)

At a tax exactly equal to the shadow price, every mix between the capped solution and "all gas" has the same total cost including the tax, and the solver returns one corner of that set. A tax slightly above the shadow price removes all coal. The cap and the tax therefore lead to the same *marginal* decision, but in a linear model the quantity can jump at the critical price. This knife-edge behaviour is a property of linear models; in large models with many technologies it is much less visible.

**💬 Question.** The exercise above illustrates the equivalence of a quantity instrument (an emission limit, or cap) and a price instrument (a tax). Under what conditions does this equivalence hold, and why might it break down in real policy?

*Answer:* *(write the answer here)*

---
# Part 4 · Algebraic modelling with Pyomo

Writing a model as a matrix, as above, becomes impractical beyond a handful of variables. Large models are written in an **algebraic modelling language**, in which sets, parameters, variables, and constraints are declared with names, and the computer builds the matrix. MESSAGEix is written in the modelling language GAMS; this series uses **Pyomo**, which has the same logic and runs in Python.

The cell below writes the three-plant problem of Part 3 in Pyomo. Read it from top to bottom: each block corresponds to one row of the table in Part 1.

In [ ]:
plants = ["coal", "gas", "wind"]
run_cost = {"coal": 60, "gas": 80, "wind": 20}      # USD/kWa
max_out = {"coal": 45, "gas": 50, "wind": 30}       # GWa
emission_factor = {"coal": 8.0, "gas": 3.4, "wind": 0.0}   # t CO2/kWa
demand, emission_cap = 50, 100


def dispatch_model(demand, emission_cap):
    m = pyo.ConcreteModel()
    # Set
    m.P = pyo.Set(initialize=plants)
    # Decision variables
    m.ACT = pyo.Var(m.P, within=pyo.NonNegativeReals)
    # Objective
    m.OBJ = pyo.Objective(expr=sum(run_cost[p] * m.ACT[p] for p in m.P), sense=pyo.minimize)
    # Constraints
    m.DEMAND = pyo.Constraint(expr=sum(m.ACT[p] for p in m.P) >= demand)
    m.LIMIT = pyo.Constraint(m.P, rule=lambda m, p: m.ACT[p] <= max_out[p])
    m.EMISSION = pyo.Constraint(expr=sum(emission_factor[p] * m.ACT[p] for p in m.P) <= emission_cap)
    # Ask the solver to return shadow prices
    m.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)
    return m


m = dispatch_model(demand, emission_cap)
solver.solve(m)
print({p: round(pyo.value(m.ACT[p]), 2) for p in plants}, "| cost", round(pyo.value(m.OBJ)))
print("electricity price:", round(m.dual[m.DEMAND], 2), "| carbon price:", round(-m.dual[m.EMISSION], 2))

The result is identical to Part 3. The advantage of the algebraic form is that the model is described by **sets and rules**, so that adding a plant only means adding an entry to the data, not rewriting the matrix. In Pyomo, the shadow price of a "less than or equal" constraint in a minimisation is negative, so it is reversed for the carbon price.

### ✏️ Exercise 4: Add a solar plant

Add a solar plant with running cost **15** USD/kWa, maximum output **10** GWa, and no emissions. Only the data need to change; the function `dispatch_model` stays as it is. Solve the model and store it as `m_solar`.

In [ ]:
plants = ["coal", "gas", "wind", ___]
run_cost[___] = ___
max_out[___] = ___
emission_factor[___] = ___

m_solar = dispatch_model(demand, emission_cap)
solver.solve(m_solar)
print({p: round(pyo.value(m_solar.ACT[p]), 2) for p in plants})
print("carbon price:", round(-m_solar.dual[m_solar.EMISSION], 2))

In [ ]:
check("solar runs at its maximum", lambda: abs(pyo.value(m_solar.ACT["solar"]) - 10) < 1e-6)
check("the emission limit is no longer binding (carbon price 0)", lambda: abs(m_solar.dual[m_solar.EMISSION]) < 1e-6)

**💬 Question.** Explain why the carbon price falls to zero when solar is added, even though the emission limit has not changed.

*Answer:* *(write the answer here)*

---
# Part 5 · A capacity expansion model over several decades

The dispatch problem took the power plants as given. Long-term planning must also decide **which plants to build**, and that decision has consequences for decades. MESSAGE therefore distinguishes three kinds of quantity, which recur in every later notebook:

| MESSAGEix name | Meaning | Unit in this notebook |
|---|---|---|
| `CAP_NEW[t, y]` | New capacity of technology *t* built per year in period *y* | GW per year |
| `CAP[t, y]` | Installed capacity in period *y*: all past `CAP_NEW` that has not yet reached the end of its technical lifetime | GW |
| `ACT[t, y]` | Activity (output) in period *y* | GWa per year |

Two constraints connect them:

$$
\text{CAP}_{t,y} = \sum_{v \leq y,\; y - v < \text{lifetime}_t} \text{duration}_v \cdot \text{CAP\_NEW}_{t,v} \qquad \text{(capacity maintenance)}
$$

$$
\text{ACT}_{t,y} \leq \text{capacity\_factor}_t \cdot \text{CAP}_{t,y} \qquad \text{(capacity constraint)}
$$

The capacity factor is the average share of the year in which a plant can operate at full output: about 0.85 for a coal plant, and about 0.3 for a wind farm, because the wind does not always blow.

Costs now have three parts: the **investment cost** (`inv_cost`, USD per kW of new capacity), the **fixed operation and maintenance cost** (`fix_cost`, USD per kW of installed capacity per year), and the **variable cost** (`var_cost`, USD per kWa of output, including fuel). Costs in later periods are **discounted** at an interest rate *r*, so that a dollar spent in 2050 counts for less than a dollar spent in 2030:

$$
\min \sum_y \text{df}_y \Big[\sum_t \text{inv\_cost}_t \,\text{CAP\_NEW}_{t,y} + \text{fix\_cost}_t \,\text{CAP}_{t,y} + \text{var\_cost}_t \,\text{ACT}_{t,y}\Big],
\qquad \text{df}_y = \sum_{k=0}^{9} (1+r)^{-(y - k - 2020)}
$$

where the discount factor df sums over the ten years of each period.

In [ ]:
years = [2030, 2040, 2050, 2060]
duration = 10
techs = ["coal", "gas", "wind"]

inv_cost = {"coal": 1500, "gas": 900, "wind": 1400}   # USD/kW
fix_cost = {"coal": 40, "gas": 25, "wind": 40}        # USD/kW/yr
var_cost = {"coal": 220, "gas": 380, "wind": 0}       # USD/kWa (mostly fuel)
cap_factor = {"coal": 0.85, "gas": 0.85, "wind": 0.35}
lifetime = {"coal": 30, "gas": 30, "wind": 25}        # years
emission_factor = {"coal": 8.0, "gas": 3.4, "wind": 0.0}   # t CO2/kWa
demand = {2030: 100, 2040: 120, 2050: 140, 2060: 160}       # GWa

# Existing capacity: 100 GW of coal built in 2020, which retires after 30 years
historical_cap = {"coal": (2020, 100.0)}

table = pd.DataFrame({"inv_cost": inv_cost, "fix_cost": fix_cost, "var_cost": var_cost,
                      "capacity_factor": cap_factor, "lifetime": lifetime})
table

The variable cost of coal (220 USD/kWa, about 25 USD/MWh) and gas (380 USD/kWa) is mostly fuel. A useful first comparison is the **levelised cost of electricity (LCOE)**: the investment cost spread over the lifetime with a capital recovery factor, plus fixed cost, divided by annual output, plus variable cost.

In [ ]:
def lcoe(t, r=0.05):
    crf = r / (1 - (1 + r) ** -lifetime[t])          # capital recovery factor
    annual_capital = crf * inv_cost[t] + fix_cost[t]  # USD per kW per year
    return annual_capital / cap_factor[t] + var_cost[t]   # USD per kWa


pd.Series({t: lcoe(t) for t in techs}, name="LCOE (USD/kWa)").round(0)

New coal and new wind have similar levelised costs with these assumptions, and gas is the most expensive because of its fuel. A levelised cost comparison, however, cannot say *when* to build, and it ignores the existing coal plants, whose investment cost has already been paid. The capacity expansion model below decides the timing and the mix, taking into account existing capacity, technical lifetimes, and discounting. It is a small version of MESSAGE.

In [ ]:
def expansion_model(r=0.05, carbon_tax=0.0, emission_cap=None):
    m = pyo.ConcreteModel()
    m.T = pyo.Set(initialize=techs)
    m.Y = pyo.Set(initialize=years)
    m.ACT = pyo.Var(m.T, m.Y, within=pyo.NonNegativeReals)
    m.CAP_NEW = pyo.Var(m.T, m.Y, within=pyo.NonNegativeReals)

    def alive(t, v, y):
        return v <= y and y - v < lifetime[t]

    # Capacity maintenance: existing plus surviving new capacity
    def cap(m, t, y):
        hist = 0.0
        if t in historical_cap and alive(t, historical_cap[t][0], y):
            hist = historical_cap[t][1]
        return hist + sum(duration * m.CAP_NEW[t, v] for v in years if alive(t, v, y))
    m.CAP = pyo.Expression(m.T, m.Y, rule=cap)

    m.COMMODITY_BALANCE = pyo.Constraint(m.Y, rule=lambda m, y: sum(m.ACT[t, y] for t in m.T) >= demand[y])
    m.CAPACITY_CONSTRAINT = pyo.Constraint(m.T, m.Y, rule=lambda m, t, y: m.ACT[t, y] <= cap_factor[t] * m.CAP[t, y])
    m.EMISS = pyo.Expression(m.Y, rule=lambda m, y: sum(emission_factor[t] * m.ACT[t, y] for t in m.T))
    if emission_cap is not None:   # cap on cumulative emissions over the horizon (Mt)
        m.EMISSION_CONSTRAINT = pyo.Constraint(expr=sum(duration * m.EMISS[y] for y in m.Y) <= emission_cap)

    df = {y: sum((1 + r) ** -(y - k - 2020) for k in range(duration)) for y in years}
    m.OBJ = pyo.Objective(sense=pyo.minimize, expr=sum(df[y] * (
        sum(inv_cost[t] * m.CAP_NEW[t, y] + fix_cost[t] * m.CAP[t, y] + var_cost[t] * m.ACT[t, y] for t in m.T)
        + carbon_tax * m.EMISS[y]) for y in m.Y))
    m.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)
    solver.solve(m)
    return m


def results(m):
    act = pd.DataFrame({t: [pyo.value(m.ACT[t, y]) for y in years] for t in techs}, index=years)
    cap = pd.DataFrame({t: [pyo.value(m.CAP[t, y]) for y in years] for t in techs}, index=years)
    new = pd.DataFrame({t: [pyo.value(m.CAP_NEW[t, y]) for y in years] for t in techs}, index=years)
    return act, cap, new


m0 = expansion_model()
act0, cap0, new0 = results(m0)
print("Activity (GWa)"); print(act0.round(1))
print("\nInstalled capacity (GW)"); print(cap0.round(1))

**💬 Predict before running the next cell.** The 100 GW of coal built in 2020 reaches the end of its 30-year lifetime in 2050. What happens to coal generation in 2050, and what replaces it?

*Answer:* *(write the answer here)*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
act0.plot.bar(stacked=True, ax=axes[0], color=["dimgrey", "tab:orange", "tab:green"], title="Activity (GWa)")
cap0.plot.bar(stacked=True, ax=axes[1], color=["dimgrey", "tab:orange", "tab:green"], title="Installed capacity (GW)")
plt.tight_layout()
plt.show()

The capacity chart shows a feature of every capacity expansion model: much more wind **capacity** is installed than its share of **activity** suggests, because each GW of wind delivers only 0.32 GWa per year.

### ✏️ Exercise 5: The effect of the discount rate

Wind is capital-intensive (high investment cost, no fuel cost), while coal and gas spend more on fuel later. Run the model with a discount rate of **2%** (`m_low`) and of **10%** (`m_high`). Compute the share of wind in total activity over all periods for each, and store them as `wind_low` and `wind_high`.

In [ ]:
m_low = expansion_model(r=___)
m_high = expansion_model(r=___)

def wind_share(m):
    act, _, _ = results(m)
    return act["wind"].sum() / act.values.sum()

wind_low = wind_share(m_low)
wind_high = ___
print(f"wind share: {wind_low:.1%} at 2%, {wind_high:.1%} at 10%")

In [ ]:
check("a lower discount rate does not reduce the wind share", lambda: wind_low >= wind_high - 1e-9)

**💬 Question.** The discount rate in an energy model is sometimes described as a technical parameter and sometimes as an ethical choice. Using the result of the exercise, explain why the choice matters for climate policy in a country such as Pakistan, where the cost of capital is high.

*Answer:* *(write the answer here)*

---
# Part 6 · A marginal abatement cost curve

The **marginal abatement cost (MAC) curve** shows how the carbon price rises as the emission target tightens. It is constructed by solving the model repeatedly with progressively stricter limits and recording the shadow price of the emission constraint each time. Integrated assessment models use exactly this procedure to explore how costly a given climate target is.

In [ ]:
baseline_emissions = sum(duration * pyo.value(m0.EMISS[y]) for y in years)
print(f"Cumulative baseline emissions 2021-2060: {baseline_emissions:,.0f} Mt CO2")

def carbon_price_2030(m, r=0.05):
    # The shadow price of the cumulative cap is in present-value USD per tonne. A tax in
    # period y with the same effect satisfies tax_y * df_y = shadow price * duration.
    df_2030 = sum((1 + r) ** -(2030 - k - 2020) for k in range(duration))
    return -m.dual[m.EMISSION_CONSTRAINT] * duration / df_2030


rows = []
for share in np.linspace(1.0, 0.2, 9):
    cap_mt = share * baseline_emissions
    m = expansion_model(emission_cap=cap_mt)
    rows.append({"reduction (%)": 100 * (1 - share),
                 "carbon price (USD/t, 2030 value)": carbon_price_2030(m),
                 "total cost (billion USD)": pyo.value(m.OBJ) / 1e3})
mac = pd.DataFrame(rows)
mac.round(1)

In [ ]:
fig, ax = plt.subplots()
ax.step(mac["reduction (%)"], mac["carbon price (USD/t, 2030 value)"], where="post", marker="o")
ax.set(xlabel="reduction of cumulative CO2 relative to baseline (%)",
       ylabel="carbon price in 2030 (USD per t CO2)", title="Marginal abatement cost curve")
plt.show()

The curve rises in **steps**. Each flat segment corresponds to one abatement option (for example, replacing coal by gas, or by wind), which is used until it is exhausted; the next, more expensive option then sets the price. In a large model with hundreds of technologies, the steps become so small that the curve looks smooth.

The price reported is the value in 2030. Because the cap is on **cumulative** emissions, the undiscounted carbon price grows over time at the discount rate (the *Hotelling rule*): a tonne avoided in 2060 is worth the same in present-value terms as a tonne avoided in 2030. Notebook 04 shows this pattern in `PRICE_EMISSION`.

### ✏️ Exercise 6: A carbon tax that reproduces the cap · ★ Stretch

Pick the cap with a **50%** reduction. Solve it (`m_cap`), read the carbon price in 2030 from its shadow price (as in the cell that built `mac`), and then solve the model with **no cap** but a carbon tax that grows at 5% per year from that 2030 value. The model function applies one tax value to all years, so use the present-value form: a constant present-value tax equals the shadow price of the cumulative cap divided by the period duration. Store the tax model as `m_tax` and compare cumulative emissions.

In [ ]:
cap_mt = 0.5 * baseline_emissions
m_cap = expansion_model(emission_cap=cap_mt)
lam = -m_cap.dual[m_cap.EMISSION_CONSTRAINT]       # shadow price of the cumulative cap

# A tax rising at the discount rate has a constant present value; expansion_model
# multiplies its carbon_tax by the discount factor, so we build the tax per year here:
def expansion_model_hotelling(lam):
    m = expansion_model()
    m.del_component(m.OBJ)
    df = {y: sum((1.05) ** -(y - k - 2020) for k in range(duration)) for y in years}
    m.OBJ = pyo.Objective(sense=pyo.minimize, expr=sum(df[y] * (
        sum(inv_cost[t] * m.CAP_NEW[t, y] + fix_cost[t] * m.CAP[t, y] + var_cost[t] * m.ACT[t, y] for t in techs))
        + ___ * duration * m.EMISS[y] for y in years))
    solver.solve(m)
    return m

m_tax = expansion_model_hotelling(lam)
em_cap = sum(duration * pyo.value(m_cap.EMISS[y]) for y in years)
em_tax = sum(duration * pyo.value(m_tax.EMISS[y]) for y in years)
print(f"cumulative emissions: cap {em_cap:,.0f} Mt | tax {em_tax:,.0f} Mt")

In [ ]:
check("the tax keeps emissions at or below the cap", lambda: em_tax <= em_cap * 1.001)

---
# Part 7 · Summary: from this notebook to MESSAGE

Every element of the capacity expansion model in Part 5 has a counterpart in MESSAGEix, with the same name:

| This notebook | MESSAGEix | Notebook where it reappears |
|---|---|---|
| `ACT`, `CAP_NEW`, `CAP` | Variables `ACT`, `CAP_NEW`, `CAP` | 04, 05, 07 |
| Demand constraint | `COMMODITY_BALANCE`, with a shadow price reported as `PRICE_COMMODITY` | 04, 05, 06 |
| Capacity constraint, capacity factor | `CAPACITY_CONSTRAINT`, parameter `capacity_factor` | 04 |
| Lifetime, historical capacity | `technical_lifetime`, `historical_new_capacity` | 04 |
| Costs | `inv_cost`, `fix_cost`, `var_cost`, `interestrate` | 04 |
| Emission limit and its shadow price | `bound_emission`, `EMISSION_CONSTRAINT`, `PRICE_EMISSION` | 04, 05 |
| Carbon tax | `tax_emission` | 04, 05 |

What MESSAGE adds is mostly **structure**: technologies take inputs and produce outputs on several **levels** (primary, secondary, final, useful energy), linked in a *reference energy system*; growth constraints limit how fast technologies can expand; and the model is driven by demands for energy services rather than for electricity alone. Notebook 04 builds exactly that structure for Westeros.

**💬 Final question.** In two or three sentences, describe one decision that a least-cost model would make differently from a real electricity sector, and explain why.

*Answer:* *(write the answer here)*

---
# References and further reading

- Huppmann, D., Gidden, M., Fricko, O., Kolp, P., Orthofer, C., Pimmer, M., et al. (2019). The MESSAGEix Integrated Assessment Model and the ix modeling platform (ixmp): An open framework for integrated and cross-cutting analysis of energy, climate, the environment, and sustainable development. *Environmental Modelling & Software*, 112, 143–156. [doi:10.1016/j.envsoft.2018.11.012](https://doi.org/10.1016/j.envsoft.2018.11.012)
- [MESSAGEix documentation: mathematical formulation](https://docs.messageix.org/en/latest/model/MESSAGE/index.html). The equation names used in this notebook follow this documentation.
- Weitzman, M. L. (1974). Prices vs. quantities. *The Review of Economic Studies*, 41(4), 477–491. [doi:10.2307/2296698](https://doi.org/10.2307/2296698)
- Hotelling, H. (1931). The economics of exhaustible resources. *Journal of Political Economy*, 39(2), 137–175. [doi:10.1086/254195](https://doi.org/10.1086/254195)
- [Pyomo documentation](https://pyomo.readthedocs.io/) and [HiGHS](https://highs.dev/), the open-source solver used throughout this series.